In [1]:
# used to load existing models
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
model_path=r"E:\AI\models\gemma-4-e4b-it"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [4]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

import transformers
print(transformers.__version__)
print(transformers.is_torch_available())

import sys
print(sys.executable)

print("Is CUDA available?:", torch.cuda.is_available())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(0))

2.11.0+cu128
True
5.12.1
True
c:\Users\hxsin\OneDrive\Desktop\Coding\Intern\Intern II\FIR-generator\.venv\Scripts\python.exe
Is CUDA available?: True
Current device: 0
Device name: NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
device

'cuda'

In [7]:
# used to load auto regressive(causal) models
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16
).to(device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

: 

In [ ]:
print(model.hf_device_map)

{'model.vision_tower.patch_embedder': 'cpu', 'model.vision_tower.encoder.rotary_emb': 'cpu', 'model.vision_tower.encoder.layers.0': 'cpu', 'model.vision_tower.encoder.layers.1': 'disk', 'model.vision_tower.encoder.layers.2': 'disk', 'model.vision_tower.encoder.layers.3': 'disk', 'model.vision_tower.encoder.layers.4': 'disk', 'model.vision_tower.encoder.layers.5': 'disk', 'model.vision_tower.encoder.layers.6': 'disk', 'model.vision_tower.encoder.layers.7': 'disk', 'model.vision_tower.encoder.layers.8': 'disk', 'model.vision_tower.encoder.layers.9': 'disk', 'model.vision_tower.encoder.layers.10': 'disk', 'model.vision_tower.encoder.layers.11': 'disk', 'model.vision_tower.encoder.layers.12': 'disk', 'model.vision_tower.encoder.layers.13': 'disk', 'model.vision_tower.encoder.layers.14': 'disk', 'model.vision_tower.encoder.layers.15': 'disk', 'model.vision_tower.pooler': 'disk', 'model.language_model': 'disk', 'lm_head': 'disk', 'model.audio_tower': 'disk', 'model.embed_vision': 'disk', 'mo

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(next(model.parameters()).device)

True
NVIDIA GeForce RTX 3050 Laptop GPU
cpu


In [ ]:
prompt = "What is theft"

In [ ]:
# created chat prompt
messages =[
    {
        'role':'user',
        'content':'What is theft?'
    }
]

In [ ]:
# gemma 4 has a specific prompt format
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt = True
)
print(formatted_prompt)
print(type(formatted_prompt))

<bos><|turn>user
What is theft?<turn|>
<|turn>model

<class 'str'>


In [ ]:
# convert prompts to token
inputs = tokenizer(formatted_prompt, return_tensors="pt") # pt = PyTorch
inputs = inputs.to(model.device)

print(list(inputs.keys()))

print(inputs)
print(len(inputs))
print(type(inputs))
print(inputs.input_ids)
print(inputs.input_ids.shape)
print(inputs.attention_mask)

['input_ids', 'attention_mask']
{'input_ids': tensor([[     2,    105,   2364,    107,   3689,    563,  34047, 236881,    106,
            107,    105,   4368,    107]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
2
<class 'transformers.tokenization_utils_base.BatchEncoding'>
tensor([[     2,    105,   2364,    107,   3689,    563,  34047, 236881,    106,
            107,    105,   4368,    107]])
torch.Size([1, 13])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


Tokenization Steps
1. Split text into tokens
-> Gemma uses sentencepiece tokenizer , breaks words into subwords

2. Creates tokens and then add special tokens(BOS,user,turn,etc.)
3. Pay attention to all tokens(if 1)

In [ ]:
print(tokenizer.convert_ids_to_tokens(inputs.input_ids[0]))

['<bos>', '<|turn>', 'user', '\n', 'What', '▁is', '▁theft', '?', '<turn|>', '\n', '<|turn>', 'model', '\n']


In [ ]:
# phase 4 -> only inference no training 
with torch.no_grad():
    output_ids=model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False, # greedy decoding -> pick highest probability token
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    '''
    for generation
    outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    top_k=40,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)
    '''

temperature controls randomness
0.1 = very deterministic, 0.7 = balanced, 1.5 =very creative

top_p -> Nucleus sampling
0.9-0.95 = common choice (sample from tokens whose cumulative probability reaches 95%)

top_k -> keep top k candidate tokens

In [ ]:
input_len = inputs['input_ids'].shape[1]
answer_only = tokenizer.decode(
    output_ids[0][input_len:],
    skip_special_tokens=True
)

In [ ]:
print("PROMPT:")
print(formatted_prompt)
print("\nANSWER:")
print(answer_only)

PROMPT:
<bos><|turn>user
What is theft?<turn|>
<|turn>model


ANSWER:
Theft is a broad concept, but in its most fundamental sense, **theft is the unlawful taking and carrying away of the personal property of another person with the intent to permanently deprive the owner of


In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50
)

# max_new_tokens > max_len since max_len limits tokens produced

In [ ]:
llm=HuggingFacePipeline(pipeline=pipe)

In [ ]:
# bype pair encoding (bpe) tokenizer - subword tokenization algo
response = llm.invoke(
        tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt = True
    )
)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
response

"<bos><|turn>user\nWhat is theft?<turn|>\n<|turn>model\nTheft is a **criminal act** that involves the **unlawful taking or appropriation of someone else's property** with the intent to permanently deprive the owner of that property.\n\nWhile the specifics can vary depending on the jurisdiction (country, state, etc.), here is a comprehensive breakdown of what theft entails:\n\n---\n\n## Core Elements of Theft\n\nFor an act to legally qualify as theft, several key elements generally must be present:\n\n**1. Taking or Appropriation:**\n* **Taking:** Physically removing the property from the owner's control.\n* **Appropriation:** Even if the item is still physically present, if the taker treats it as their own and uses it in a way that denies the rightful owner their use or ownership, it can be considered appropriation.\n\n**2. Property:**\n* The item taken must have **value**. This can be tangible property (money, electronics, jewelry) or sometimes intangible property (intellectual proper

In [ ]:
len(response)

1006

In [ ]:
# input_len = inputs['input_ids'].shape[1]
# final_response = tokenizer.decode(
#     response,
#     skip_special_tokens=True
# )
# final_response

In [ ]:
import platform
print(platform.processor())

AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD


In [ ]:
print(model.device)

NameError: name 'model' is not defined

In [ ]:
import torch

print(torch.__file__)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

c:\Users\hxsin\OneDrive\Desktop\Coding\Intern\Intern II\FIR-generator\.venv\Lib\site-packages\torch\__init__.py
2.11.0+cu128
12.8
True


In [ ]:
from ollama import chat

response = chat(
    model='gemma3:1b',
    messages=[
        {'role':'user','content':'Explain the ipc sections involved in murder?'}
    ]
)
print(response.message.content)

